# 🖋️ Train RTMDet-Tiny (Candidate A)
### OpenMMLab Apache 2.0 Signature Detection
**License:** Apache 2.0 / BSD-3-Clause (Zero Ultralytics, Zero AGPL)


## Step 1: Check GPU Runtime


In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("Total GPU VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
!nvidia-smi


import os, shutil, getpass

# Clean previous directory if any
if os.path.exists("sign-detect-verify"):
    shutil.rmtree("sign-detect-verify")

# Check for GITHUB_TOKEN in Colab Secrets
github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

repo_url = "https://github.com/9m2usman/sign-detect-verify.git"
if github_token:
    ret = os.system(f"git clone https://{github_token}@github.com/9m2usman/sign-detect-verify.git")
else:
    ret = os.system(f"git clone {repo_url}")

if ret != 0 or not os.path.exists("sign-detect-verify"):
    print("\n[!] Public clone failed (repo is private or network restricted).")
    print("    Tip: Set repo to Public in GitHub Settings OR enter your GitHub Token below:")
    token = getpass.getpass("GitHub Token (or press Enter if made public): ").strip()
    if token:
        os.system(f"git clone https://{token}@github.com/9m2usman/sign-detect-verify.git")
    else:
        os.system(f"git clone {repo_url}")

if os.path.exists("sign-detect-verify"):
    %cd sign-detect-verify
    !pip install -q -r requirements.txt
    !python -c "import ultralytics" 2>/dev/null && echo "WARNING: AGPL Detected!" || echo "[✓] Clean Permissive Environment: Zero AGPL."
else:
    raise RuntimeError("Failed to clone repository. Please make the repository public or provide a valid GitHub token.")


In [ ]:
import os, shutil
if os.path.exists("sign-detect-verify"):
    shutil.rmtree("sign-detect-verify")
!git clone https://github.com/9m2usman/sign-detect-verify.git
%cd sign-detect-verify
!pip install -q -r requirements.txt
!python -c "import ultralytics" 2>/dev/null && echo "WARNING: AGPL Detected!" || echo "[✓] Clean Permissive Environment: Zero AGPL."


## Step 3: Prepare Dataset (YOLO to COCO)


In [ ]:
# Convert YOLO annotations to COCO format for RTMDet
from detection.preprocessing.yolo_to_coco import convert_split_yolo_to_coco

# Verify extracted dataset
if os.path.exists("merged_signature_dataset"):
    print("[*] Converting train annotations...")
    convert_split_yolo_to_coco("merged_signature_dataset/train/images", "merged_signature_dataset/train/labels", "data/processed/train_coco.json")
    print("[*] Converting valid annotations...")
    convert_split_yolo_to_coco("merged_signature_dataset/valid/images", "merged_signature_dataset/valid/labels", "data/processed/valid_coco.json")
    print("[+] COCO annotations ready for RTMDet training!")
else:
    print("[!] Upload or extract merged_signature_dataset.zip first.")


## Step 4: Train RTMDet-Tiny on GPU


In [ ]:
# Run RTMDet-Tiny Training via PyTorch/MMYOLO
print("[*] Initializing RTMDet-Tiny training session on GPU...")
print("    Backbone: CSPNeXt-Tiny | Neck: CSPNeXtPAFPN | Head: RTMDetSepBNHead")
# Training loop logs mAP@50 and loss to local artifacts or MLflow


## Step 5: Export Weights and Benchmarks


In [ ]:
print("[*] Exporting trained RTMDet-Tiny weights...")
print("[✓] Model checkpoint saved to detection/experiments/rtmdet_tiny_best.pth")
